# Multimodal
Multimodality refers to the ability to work with data that comes in different forms, such as text, audio, images, and video. LangChain includes standard types for these data that can be used across providers.

[Chat models](https://docs.langchain.com/oss/python/langchain/models) can accept multimodal data as input and generate it as output.

In general, those models accept [messages inputs](https://docs.langchain.com/oss/python/langchain/messages#multimodal) like images, PDFs, audio, or videos already converted to Base64. Some model providers can accept URLs, the file itself, or a provider-managed file ID.

Below we show short examples of input messages featuring multimodal data.

## Import Libraries

In [3]:
import base64
from dotenv import load_dotenv
from langchain.agents import create_agent
from ipywidgets import FileUpload
from IPython.display import display
from langchain.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(override=True)

True

## Image input

In [ ]:
# Ollama only supported through in-line base64 format.
agent = create_agent(
    "ollama:qwen3.5:4b",
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)


uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [2]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x10f846440>, 'last_modified': datetime.datetime(2026, 11, 12, 19, 10, 16, 850000, tzinfo=datetime.timezone.utc)},)


### From base64 data

In [3]:
# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

As an AI science fiction writer, I have imagined this capital for you based on the visual evidence of the landscape and architecture. Here is a profile for **The Azure Ridge**.

### **Name:** The Azure Ridge (Capital of the Kepler Hegemony)
**Location:** Perched upon the jagged, precipice of a colossal impact crater in the system's outer rim.

**Visual Description:**
*   **Verticality and Architecture:** The capital is defined by its incredible height. It isn't a sprawling sprawl like Earth; it is a forest of spires that pierce the rocky horizon. The architecture is sleek, futuristic, and organic—tall, columnar structures resembling crystallized steel or white quartz, glowing with an intense cyan energy that pulses in rhythm with the city's life support systems.
*   **The Glow:** In a barren world of rust-colored rocks and grey crags, the blue light is the dominant feature. This glow suggests a civilization built on high-energy density—perhaps harvesting zero-point energy from the rock

Ollama does not currently support image URLs. 

You need to obtain them beforehand, as we did with the Moon Image. 

In [ ]:
from ollama import chat

response = chat(
    model='qwen2.5vl:7b',
    messages=[
        {
            'role': 'user',
            'content': 'Describe the content of this image.',
            'images': ['sandbox/moon.png'],
        }
    ],
)
print(response.message.content)

The image depicts a futuristic landscape with a large, blue planet dominating the sky on the left side. The foreground features rugged, rocky terrain with a few glowing blue patches, possibly indicating water or energy sources. In the background, there is a cityscape with tall, slender buildings that have a modern, sci-fi design, featuring vertical lines and illuminated sections. The overall scene suggests an alien world with advanced technology and a striking contrast between the natural and constructed environments.


At LangChain, it still asking for Base 64.

From other model providers that accept image URLs, you can go with:

In [5]:
import os

# 1. Prevent accidental fallback to Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "false"

# 2. Initialize the free-tier Gemini model explicitly with your AI Studio key
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    api_key=os.getenv("GEMINI_API_KEY"),
    vertexai=False,
)

agent = create_agent(
    model=llm,
    system_prompt="You are a computer specialist writer.",
)

multimodal_question = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "url": "http://vitarts3.hospedagemdesites.ws/wp-content/uploads/2026/08/Shuffle.jpeg"},
    ]
}

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## Audio input

In [10]:
import io
import time
import base64
import sounddevice as sd
from scipy.io.wavfile import write
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100
print("Hi, start recording a short and clear instruction for the LLM to execute, something like: \n`Write me a poem about cats.`\n")
print("-"*60)
print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Hi, start recording a short and clear instruction for the LLM to execute, something like: 
`Write me a poem about cats.`

------------------------------------------------------------
Recording...


100%|██████████| 50/50 [00:05<00:00,  9.58it/s]


Done.


In [9]:
agent = create_agent(
    "google_genai:gemini-2.5-flash",
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Following the instruction on the audio."},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Here's a poem about cats:

A shadow in the sun, a purr in the night,
With eyes of emerald, gleaming so bright.
They stalk with a grace, a silent soft tread,
From tip of their tail to the top of their head.

A twitch of a whisker, a flick of an ear,
They stretch and they yawn, banishing all fear.
Then curl in a sunbeam, a warm furry ball,
Observing their kingdom, surveying it all.

From playful pounce on a dangling toy string,
To a dignified nap, the peace that they bring.
They rub on your leg, a gentle soft plea,
For a scratch behind ears, or a treat, fancy free.

Mysterious creatures, with spirits so grand,
The rulers of laps, throughout every land.
With independent charm, and a knowing soft gaze,
We're utterly smitten, in so many ways.
